In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

# LSTM 方向予測 v2（lstm_dir_v2）学習ノートブック

USDJPY 5分足方向予測 - マルチタイムフレーム特徴量版

## 前回実験（lstm_dir）との違い
- **前回**: M5 の 16特徴量のみ → 方向正答率 50.3%（コイントス）
- **今回**: M5 16列 + M15・H1 上位足コンテキスト 8列 = **24特徴量**
  - M15: ma20_deviation, rsi, hlo_ratio, atr_ratio
  - H1 : ma20_deviation, rsi, hlo_ratio, bb_pband

## 学習データ
- 入力: `pearless-usdjpy-m5-mtf` dataset の npy（24列）
- フィルタ: UP(0)/DOWN(1) サンプルのみ（NEUTRAL 除外）
- 目標: val_f1_updown の最大化

## 出力
- `/kaggle/working/best_model.pt`
- `/kaggle/working/logs/training_log_lstm_dir_v2_{timestamp}.csv`

In [ ]:
import subprocess, sys

# Kaggle プリインストールの torch (2.10+cu128) は P100 (sm_60) のカーネルイメージを
# 含まず "CUDA error: no kernel image is available" で学習が落ちる（2026-06-11 実測）。
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
import torch
assert torch.__version__.startswith("2.5.1"), f"torch downgrade failed: {torch.__version__}"
print(f"torch {torch.__version__}")

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

# MTF npy dataset（X_train_mtf.npy など 24列）
MTF_DATASET_DIR = Path("/kaggle/input/datasets/nomuhosokawa/pearless-usdjpy-m5-mtf")
if not MTF_DATASET_DIR.exists():
    MTF_DATASET_DIR = Path("/kaggle/input/pearless-usdjpy-m5-mtf")
if not MTF_DATASET_DIR.exists():
    MTF_DATASET_DIR = Path("data/npy_mtf")  # ローカルフォールバック

WORKING_DIR = Path("/kaggle/working")

# ソースコードを sys.path に追加
for _repo_candidate in [
    Path("/kaggle/input/pearless-src"),
    Path("/kaggle/input/datasets/nomuhosokawa/pearless-src"),
]:
    if _repo_candidate.exists():
        REPO_ROOT = _repo_candidate
        break
else:
    REPO_ROOT = Path("..")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"MTF Dataset dir: {MTF_DATASET_DIR}")
print(f"Working dir: {WORKING_DIR}")

In [ ]:
# MTF npy の読み込み（24特徴量）
X_train: np.ndarray = np.load(MTF_DATASET_DIR / "X_train_mtf.npy")
y_train: np.ndarray = np.load(MTF_DATASET_DIR / "y_train.npy")
X_val: np.ndarray   = np.load(MTF_DATASET_DIR / "X_val_mtf.npy")
y_val: np.ndarray   = np.load(MTF_DATASET_DIR / "y_val.npy")

print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}")

assert X_train.ndim == 3 and X_train.shape[1] == 60 and X_train.shape[2] == 24, (
    f"X_train shape 不正（24列期待）: {X_train.shape}"
)

unique, counts = np.unique(y_train, return_counts=True)
print("\nクラス分布 (y_train):")
for cls, cnt in zip(unique, counts):
    label = ["UP", "DOWN", "NEUTRAL"][cls]
    print(f"  {label}({cls}): {cnt} ({cnt / len(y_train) * 100:.1f}%)")

In [ ]:
# Stage 2 フィルタ: UP(0) / DOWN(1) のみ残す（NEUTRAL(2) 除外）
train_mask = y_train != 2
val_mask   = y_val   != 2
X_train, y_train = X_train[train_mask], y_train[train_mask]
X_val,   y_val   = X_val[val_mask],     y_val[val_mask]

for name, y in [("y_train", y_train), ("y_val", y_val)]:
    unique, counts = np.unique(y, return_counts=True)
    dist = {int(u): int(c) for u, c in zip(unique, counts)}
    print(f"{name}: n={len(y)}, クラス分布={dist}")

assert set(np.unique(y_train)) == {0, 1}, "UP/DOWN 以外が残っている"

In [ ]:
# スモークテスト（本番実行時は False）
SMOKE_TEST = False

if SMOKE_TEST:
    X_train = X_train[:500]
    y_train = y_train[:500]
    X_val   = X_val[:100]
    y_val   = y_val[:100]
    print("[SMOKE] データを縮小: X_train", X_train.shape, "X_val", X_val.shape)

In [ ]:
# lstm_dir_v2 モデルの初期化
from models.configs import MODEL_CONFIGS

MODEL_NAME = "lstm_dir_v2"
config = MODEL_CONFIGS[MODEL_NAME]
print(f"使用特徴量数: {config.n_features}")
print(f"M5特徴量: {list(config.features[:16])}")
print(f"MTF特徴量: {list(config.features[16:])}")
print(f"学習ハイパラ: {config.train}")

model = config.build_model()

total_params = sum(p.numel() for p in model.parameters())
assert total_params <= 10_000_000, f"パラメータ数が 10M 超: {total_params:,}"
print(f"パラメータ数: {total_params:,} (≤ 10M OK)")

model.eval()
with torch.no_grad():
    x_test = torch.randn(4, 60, config.n_features)
    output = model(x_test)
assert output.shape == (4, 3), f"shape 不正: {output.shape}"
print(f"forward 契約 OK: output.shape={output.shape}")

In [ ]:
# 学習実行
# X_train / X_val は 24列の MTF npy をそのまま渡す
# config.features == ALL_FEATURES（24列）なので select_features は pass-through
from models.training import train_from_config

model = train_from_config(
    config,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    checkpoint_dir=str(WORKING_DIR),
    log_dir=str(WORKING_DIR / "logs"),
    n_epochs=3 if SMOKE_TEST else None,
    patience=3 if SMOKE_TEST else None,
)

In [ ]:
# チェックポイント・ログ確認
best_model_path = WORKING_DIR / "best_model.pt"
assert best_model_path.exists(), f"best_model.pt が存在しない: {best_model_path}"
print(f"チェックポイント確認 OK: {best_model_path}")

model_loaded = config.build_model()
model_loaded.load_state_dict(torch.load(best_model_path, map_location="cpu"))
model_loaded.eval()

with torch.no_grad():
    x_verify = torch.randn(2, 60, config.n_features)
    output_verify = model_loaded(x_verify)

assert output_verify.shape == (2, 3), f"ロード後 shape 不正: {output_verify.shape}"
print(f"モデルロード後の推論確認 OK: shape={output_verify.shape}")

log_dir = WORKING_DIR / "logs"
log_files = list(log_dir.glob(f"training_log_{MODEL_NAME}_*.csv"))
assert len(log_files) > 0, f"ログファイルが見つからない: {log_dir}"
print(f"ログファイル確認 OK: {log_files[0]}")

# val 方向正答率を最終確認
import pandas as pd
log_df = pd.read_csv(log_files[0])
best_epoch = log_df.loc[log_df["val_f1_up"] + log_df["val_f1_down"], :].index[0] if "val_f1_up" in log_df.columns else 0
best_f1_updown = ((log_df["val_f1_up"] + log_df["val_f1_down"]) / 2).max()
best_acc = log_df["val_accuracy"].max()
print(f"\n=== 実験結果サマリ ===")
print(f"best val_f1_updown : {best_f1_updown:.4f}")
print(f"best val_accuracy  : {best_acc:.4f}")
print(f"エポック数         : {len(log_df)}")
print(f"\n全確認完了。")